Import dataset and libraries


In [ ]:
import pandas as pd
import tqdm
import numpy as np
import torch
from torch import nn
import os

In [13]:
!rm rgbd-dataset_eval.zip

!curl -L -A "Mozilla/5.0" \
https://rgbd-dataset.cs.washington.edu/dataset/rgbd-dataset_eval/rgbd-dataset_eval.zip \
-o rgbd-dataset_eval.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  642M  100  642M    0     0  59.4M      0  0:00:10  0:00:10 --:--:-- 81.4M


In [18]:
!unzip -q rgbd-dataset_eval.zip
!ls -l

total 657688
drwxr-xr-x 53 root root      4096 Oct 17  2012 rgbd-dataset_eval
-rw-r--r--  1 root root 673456874 Mar  7 03:53 rgbd-dataset_eval.zip
drwxr-xr-x  1 root root      4096 Feb  6 14:31 sample_data


Prepare data



In [22]:
from pathlib import Path
from sklearn.model_selection import train_test_split

root = Path("rgbd-dataset_eval") 

# Class folders
classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
class_to_idx = {c: i for i, c in enumerate(classes)}

# Collect paired RGB and depth paths with labels
X_rgb, X_depth, y = [], [], []

# look for RGB files and then check if the corresponding depth file exists
for cls in classes:
    cls_dir = root / cls

    rgb_paths = sorted([
        p for p in cls_dir.rglob("*_crop.png")
        if "depthcrop" not in p.name and "maskcrop" not in p.name
    ])

    # For each RGB path, check if the corresponding depth path exists
    for rgb_path in rgb_paths:
        depth_path = Path(str(rgb_path).replace("_crop.png", "_depthcrop.png"))

        if depth_path.exists():
            X_rgb.append(str(rgb_path))
            X_depth.append(str(depth_path))
            y.append(class_to_idx[cls])

# train-test split
X_rgb_train, X_rgb_test, X_depth_train, X_depth_test, y_train, y_test = train_test_split(
    X_rgb, X_depth, y, test_size=0.2, random_state=1234, stratify=y
)

print("Total paired samples:", len(y))
print("Example:", X_rgb[0], X_depth[0], y[0])
print("Train:", len(y_train), "Test:", len(y_test), "Num classes:", len(classes))

Total paired samples: 41877
Example: rgbd-dataset_eval/apple/apple_1/apple_1_1_101_crop.png rgbd-dataset_eval/apple/apple_1/apple_1_1_101_depthcrop.png 0
Train: 33501 Test: 8376 Num classes: 51


Create RGB and depth CNN models

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, in_channels, num_classes, feat_dim=256):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), 
            nn.ReLU(), 
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), 
            nn.ReLU(), 
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), 
            nn.ReLU(), 
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), 
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, feat_dim), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(feat_dim, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.fc(x)

In [24]:
RGB_model = SmallCNN(in_channels=3, num_classes=len(classes))
Depth_model = SmallCNN(in_channels=1, num_classes=len(classes))

Train RGB and Depth CNN models using masked supervised learning 

Combine the extraction features from both models using late fusion

Test the accuracy of the model